# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dwaynemongaya/flyrank-ml-internship_dwaynemongaya/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method: Decision Tree

I will use a Decision Tree as my first model. My Week-4 baseline uses fixed thresholds on a small number of search signals. A Decision Tree can learn thresholds and interactions between several observable signals from the training data instead of requiring me to define every rule manually.

I chose this method because it remains interpretable: I can inspect which features and thresholds the model uses and compare its ranking performance directly with my Week-4 baseline. I will keep the tree relatively simple because the goal is to test whether learned patterns improve the baseline, not to reward complexity by itself.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split design: Grouped by client

I will split the data by client_hash_id so that observations from the same client do not appear in both the training and test sets. The model will learn from one group of clients and will be evaluated on different, unseen clients.

This is more honest than randomly splitting individual rows because pages from the same client may share similar search and content patterns. Allowing the same client into both sets could make the model appear better than it really is. Grouping by client therefore tests whether the learned patterns generalize beyond the clients used for training.

I will use the same test observations and evaluation metric when comparing the Decision Tree with my Week-4 baseline.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))

df.head()

Rows: 30000
Columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [4]:
import numpy as np

features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

# Observable features only
X = (
    df[features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

# Create the target from the observed outcome
y = (df["trend_direction"] == "down").astype(int)

# Used only for grouped train/test splitting
groups = df["client_id"]

print("Features:", features)
print("Target: trend_direction == 'down'")
print("Declining pages:", y.sum())
print("Non-declining pages:", (y == 0).sum())
print("Total rows:", len(y))

Features: ['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'word_count']
Target: trend_direction == 'down'
Declining pages: 16262
Non-declining pages: 13738
Total rows: 30000


In [6]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(train_clients & test_clients))

Training rows: 23837
Test rows: 6163
Training clients: 25
Test clients: 7
Client overlap: 0


In [7]:
baseline_score = (
    (X_test["impressions_90d"] >= 500).astype(int) * 50 +
    (X_test["avg_position"] <= 10).astype(int) * 50
)

In [8]:
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(
    max_depth=4,
    class_weight="balanced",
    random_state=42
)

tree.fit(X_train, y_train)

tree_score = tree.predict_proba(X_test)[:, 1]

print("Decision Tree trained.")

Decision Tree trained.


In [9]:
def precision_at_k(scores, labels, k):
    scores = np.asarray(scores)
    labels = np.asarray(labels)

    order = np.argsort(-scores)
    top_k = labels[order[:k]]

    return top_k.mean()

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [10]:
# Build a test-set review table
error_df = X_test.copy()

error_df["actual"] = y_test.values
error_df["tree_score"] = tree_score
error_df["predicted"] = (tree_score >= 0.5).astype(int)

# Keep only mistakes
errors = error_df[
    error_df["actual"] != error_df["predicted"]
]

print("Test rows:", len(error_df))
print("Wrong predictions:", len(errors))
print("Error rate:", round(len(errors) / len(error_df), 3))

display(errors.head(10))

Test rows: 6163
Wrong predictions: 2820
Error rate: 0.458


,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,word_count,actual,tree_score,predicted
1,445,25,15320,20.3,0.05,2481.0,1,0.484629,0
13,238,103,307,39.8,0.00,1342.0,0,0.688416,1
19,187,20,99,6.9,2.02,2673.0,1,0.405297,0
23,502,20,297,13.9,0.34,0.0,1,0.484629,0
25,180,20,27,7.2,0.00,2777.0,1,0.498834,0
26,300,13,2426,30.0,0.12,2686.0,0,0.688416,1
37,228,13,187,11.5,1.07,2658.0,1,0.405297,0
39,348,104,4,36.3,0.00,3666.0,1,0.138182,0
47,126,20,8,25.5,0.00,3086.0,1,0.498834,0
49,174,8,9,10.1,0.00,1585.0,1,0.498834,0


In [11]:
false_positives = errors[
    (errors["actual"] == 0) &
    (errors["predicted"] == 1)
]

false_negatives = errors[
    (errors["actual"] == 1) &
    (errors["predicted"] == 0)
]

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

False positives: 729
False negatives: 2091


In [12]:
importance_df = pd.DataFrame({
    "feature": features,
    "importance": tree.feature_importances_
}).sort_values("importance", ascending=False)

display(importance_df)

,feature,importance
2,impressions_90d,0.535267
0,content_age_days,0.231760
3,avg_position,0.102583
4,ctr,0.092535
5,word_count,0.024618
1,days_since_last_update,0.013237


### Error analysis and interpretation

On the 6,163-row test set, the Decision Tree made 2,820 incorrect classifications, giving an error rate of about 45.8%. The errors were not evenly distributed: there were 729 false positives but 2,091 false negatives. This means the model more often failed to identify pages that were actually declining than incorrectly identified non-declining pages as declining. For a review-ranking system, these false negatives are important because potentially useful review candidates could be missed.

The tree relied most strongly on impressions_90d, with a feature importance of about 0.535, followed by content_age_days at about 0.232. avg_position and ctr contributed smaller amounts, while word_count and days_since_last_update contributed relatively little in this fitted tree. These importances describe which variables the tree used for its splits; they do not show that those variables cause content decline.

Overall, the model captures some useful patterns but still makes substantial errors, especially false negatives. This suggests that a simple Decision Tree may not fully capture the relationships between the available signals and declining content, and any advantage over the baseline should be judged from the ranking metrics rather than model complexity alone.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.